# Hybrid Reddit Dating Questions Extractor v3.0

## 🎯 Revolutionary Hybrid Approach

This notebook implements a **hybrid scoring model** that learns from your actual core app questions to dramatically improve Reddit question extraction and scoring. Instead of using generic keyword matching, it analyzes the patterns in your proven high-quality questions and uses machine learning to find similar questions on Reddit.

### 🧠 How the Hybrid Model Works

1. **Analyzes Your Core Questions**: Extracts patterns, frameworks, and quality indicators from your 300+ proven questions
2. **Establishes Quality Baselines**: Uses your questions to calibrate what "high quality" means
3. **Similarity Matching**: Uses TF-IDF vectorization and cosine similarity to find Reddit questions similar to your core set
4. **Pattern Recognition**: Identifies question frameworks, emotional depth, engagement potential, and conversation flow
5. **Calibrated Scoring**: Scores questions on a 0-100 scale calibrated to your quality standards

### 🎯 Key Advantages

- **Learns from YOUR data**: Uses your actual proven questions as the gold standard
- **Much higher precision**: Finds questions that match your app's quality and style
- **Intelligent filtering**: Automatically filters out low-quality and irrelevant questions
- **Explainable scoring**: Shows exactly why each question received its score
- **Continuous improvement**: Can be retrained as you add more core questions

### 📊 Output Structure

**Required Columns**: question_id, question, theme, reddit_topic, score, timestamp

**Hybrid Scoring Breakdown**:
- `similarity_to_core`: How similar to your proven questions (0-100)
- `framework_match`: Matches proven question frameworks (0-100)
- `depth_potential`: Conversation depth potential (0-100)
- `engagement_potential`: Engagement and interest level (0-100)
- `personal_connection`: Personal sharing potential (0-100)

### 🎛️ Interactive Controls

- **Quality thresholds**: Adjust minimum scores for inclusion
- **Similarity weights**: Control how much to weight different scoring components
- **Dating relevance**: Toggle dating-specific filtering
- **Subreddit selection**: Choose which subreddits to extract from
- **Real-time analysis**: Analyze individual questions in detail

## 📦 Setup and Dependencies

In [20]:
# Install required packages
import subprocess
import sys

def install_package(package):
    """Install a package using pip"""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ Successfully installed {package}")
    except subprocess.CalledProcessError:
        print(f"❌ Failed to install {package}")

# Core dependencies for hybrid model
packages = [
    "praw>=7.0.0",
    "pandas>=1.3.0", 
    "scikit-learn>=1.0.0",
    "openpyxl>=3.0.0",
    "tqdm>=4.60.0",
    "numpy>=1.21.0"
]

print("🚀 Installing hybrid model dependencies...")
for package in packages:
    install_package(package)

print("\n🎉 Setup complete!")

🚀 Installing hybrid model dependencies...
✅ Successfully installed praw>=7.0.0
✅ Successfully installed pandas>=1.3.0
✅ Successfully installed scikit-learn>=1.0.0
✅ Successfully installed openpyxl>=3.0.0
✅ Successfully installed tqdm>=4.60.0
✅ Successfully installed numpy>=1.21.0

🎉 Setup complete!


## 🔧 Configuration and Core Questions Setup

**IMPORTANT**: Upload your core questions file and update the path below.

In [21]:
# ==============================================================================
# 🔧 STEP 1: CONFIGURE EXTRACTION PARAMETERS
# ==============================================================================

# --- MODIFIABLE EXTRACTION PARAMETERS ---
# Edit these values to control the data collection process.
EXTRACTION_CONFIG = {
    'posts_per_subreddit': 50,
    'time_filter': 'month', # Options: 'all', 'year', 'month', 'week', 'day'
}

# --- SUBREDDIT SELECTION (ENHANCED) ---
# Add or remove subreddits from these groups to change the data sources.
SUBREDDIT_GROUPS = {
    'dating_focused': [
        'dating_advice', 'dating', 'relationships', 'relationship_advice',
        'datingoverthirty', 'datingoverforty'
    ],
    'conversation_starters': [
        'AskReddit', 'CasualConversation', 'SeriousConversation',
        'socialskills'
    ],
    'personal_questions': [
        'AskWomen', 'AskMen', 'TrueAskReddit', 'DeepThoughts', 'self'
    ],
    'fun_and_icebreakers': [
        'icebreakers',
        'WouldYouRather',
        'MakeNewFriendsHere',
        'hypotheticalsituation'
    ]
}

# --- ACTIVE GROUPS ---
# Add or remove group names from this list to activate/deactivate them.
ACTIVE_SUBREDDIT_GROUPS = [
    'dating_focused',
    'conversation_starters',
    'fun_and_icebreakers'
]

# --- This part automatically builds the final list to scan ---
SUBREDDITS_TO_SCAN = [subreddit for group in ACTIVE_SUBREDDIT_GROUPS for subreddit in SUBREDDIT_GROUPS.get(group, [])]

# --- Confirmation Printouts ---
print("⚙️ Extraction configuration loaded!")
print(f"   - Active Groups: {', '.join(ACTIVE_SUBREDDIT_GROUPS)}")
print(f"   - Total subreddits to scan: {len(SUBREDDITS_TO_SCAN)}")
print(f"   - Time filter: '{EXTRACTION_CONFIG['time_filter']}'")

⚙️ Extraction configuration loaded!
   - Active Groups: dating_focused, conversation_starters, fun_and_icebreakers
   - Total subreddits to scan: 14
   - Time filter: 'month'


### Reddit Connection

In [22]:
import praw

# Initialize Reddit connection
reddit = None
try:
    # Assumes you have a praw.ini file configured
    reddit = praw.Reddit(site_name="DEFAULT", user_agent="HybridDatingQuestionScraper/1.0")
    
    # A simple test to ensure the object was created
    if reddit:
        print("✅ Successfully initialized Reddit API connection.")
    
except Exception as e:
    print(f"❌ Failed to connect to Reddit: {e}")
    print("Please ensure your praw.ini file is correctly set up in the same directory.")

# Check if the connection is read-only (which is expected and OK for scraping)
if reddit:
    is_read_only = reddit.read_only
    print(f"   - Read-only mode: {is_read_only} (This is expected and OK for scraping)")

✅ Successfully initialized Reddit API connection.
   - Read-only mode: True (This is expected and OK for scraping)


### Reddit Extractor

In [ ]:
# (This is the replacement for your "Reddit Extractor" cell)

from tqdm.notebook import tqdm
import re

class RedditQuestionExtractor:
    def __init__(self, reddit_instance):
        if reddit_instance is None: raise ValueError("A valid PRAW Reddit instance is required.")
        self.reddit = reddit_instance
        self.seen_questions = set()

    def _is_valid_question(self, text: str) -> bool:
        """
        Enhanced and corrected validator to properly distinguish questions from statements.
        """
        text = text.strip()
        if not (10 < len(text) < 250): return False

        text_lower = text.lower()
        
        # REMOVED 'if' from this tuple to prevent false positives.
        question_starters = ('what', 'how', 'why', 'when', 'who', 'which', 'would', 'is', 'are', 'do', 'does', 'have', 'can')
        prompt_starters = ('tell me about', 'describe', 'share a time', 'name your top', 'explain')

        # Standard check for prompts or questions starting with a question word.
        if text.endswith('?') or text_lower.startswith(question_starters) or text_lower.startswith(prompt_starters):
            return True
        
        # NEW: Add a special, stricter rule for sentences starting with "If".
        # This prevents advice statements from being classified as questions.
        if text_lower.startswith('if'):
            # An "if" sentence is only a question if it actually contains a question mark.
            if '?' in text:
                return True

        return False # If none of the above, it's not a valid question.

    def extract_from_subreddit(self, subreddit_name: str, posts_limit: int = 50) -> list[dict]:
        extracted_data = []
        print(f"🔭 Searching for questions in r/{subreddit_name}...")
        try:
            subreddit = self.reddit.subreddit(subreddit_name)
            top_posts = subreddit.top(time_filter="month", limit=posts_limit)

            for post in tqdm(list(top_posts), desc=f"Processing r/{subreddit_name}", leave=False):
                # Add score and comments metadata here for later use
                post_info = {
                    'subreddit': subreddit_name,
                    'reddit_topic': post.link_flair_text or 'N/A',
                    'reddit_score': post.score,
                    'reddit_comments': post.num_comments
                }

                if self._is_valid_question(post.title):
                    if post.title.lower() not in self.seen_questions:
                        item = post_info.copy()
                        item['question'] = post.title
                        extracted_data.append(item)
                        self.seen_questions.add(post.title.lower())

                if post.is_self and post.selftext:
                    clean_text = re.sub(r'(&gt;|\>)(.*)', '', post.selftext)
                    sentences = re.split(r'[.!?\n]+', clean_text)
                    for sentence in sentences:
                        s = sentence.strip()
                        if self._is_valid_question(s):
                             if s.lower() not in self.seen_questions:
                                item = post_info.copy()
                                item['question'] = s
                                extracted_data.append(item)
                                self.seen_questions.add(s.lower())
        except Exception as e:
            print(f"⚠️ Could not process r/{subreddit_name}. Reason: {e}")

        print(f"   - Found {len(extracted_data)} new potential questions.")
        return extracted_data

print("✅ RedditQuestionExtractor class is ready (with enhanced question validation).")

✅ RedditQuestionExtractor class is ready (with enhanced question validation).


## 📚 Import Libraries and Initialize Hybrid Model

In [24]:
# Core libraries
import pandas as pd
import numpy as np
import re
import time
import json
import hashlib
from datetime import datetime
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple
from collections import Counter

# Progress tracking
from tqdm.notebook import tqdm

# Machine learning for hybrid model
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Try to import optional dependencies
try:
    import praw
    PRAW_AVAILABLE = True
    print("✅ PRAW (Reddit API) available")
except ImportError:
    PRAW_AVAILABLE = False
    print("⚠️ PRAW not available - Reddit extraction disabled")

try:
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment
    EXCEL_AVAILABLE = True
    print("✅ openpyxl available for Excel formatting")
except ImportError:
    EXCEL_AVAILABLE = False
    print("⚠️ openpyxl not available - using basic Excel export")

# Set up output directory (same as original notebook)
project_root = Path.cwd()
outputs_dir = project_root / "outputs"
csv_dir = outputs_dir / "csv"
csv_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📁 Output directory: {csv_dir}")
print("🎉 All libraries imported successfully!")

✅ PRAW (Reddit API) available
✅ openpyxl available for Excel formatting

📁 Output directory: /Users/justsuyash/Documents/GitHub/AlignedV1.001/notebooks/outputs/csv
🎉 All libraries imported successfully!


## 🧠 Hybrid Scoring Model Implementation

This is the core of the hybrid approach - a machine learning model that learns from your core questions.

In [ ]:
# (This is the replacement for your "Hybrid Scoring Model" cell)

class HybridScoringModel:
    """Hybrid scoring model that learns from core app questions"""
    
    def __init__(self, core_questions: List[str]):
        self.core_questions = core_questions
        self.vectorizer = TfidfVectorizer(max_features=1500, stop_words='english', ngram_range=(1, 3), lowercase=True, min_df=1, max_df=0.9)
        self.core_vectors = self.vectorizer.fit_transform(core_questions)
        self.baselines = self._establish_baselines()
        print(f"🤖 Hybrid model initialized with {len(core_questions)} core questions")
        print(f"📊 Quality baselines established from your data")
    
    def _establish_baselines(self):
        print("🔧 Analyzing your core questions to establish quality baselines...")
        similarity_scores, pattern_scores = [], []
        sample_size = min(50, len(self.core_questions))
        sample_questions = np.random.choice(self.core_questions, sample_size, replace=False)
        for question in sample_questions:
            question_vector = self.vectorizer.transform([question])
            similarities = cosine_similarity(question_vector, self.core_vectors)[0]
            similarities = similarities[similarities < 0.99]
            if len(similarities) > 0: similarity_scores.append(np.max(similarities))
            pattern_scores.append(self._calculate_pattern_scores(question))
        baselines = {
            'similarity_baseline': np.mean(similarity_scores) if similarity_scores else 0.3,
            'pattern_baselines': {
                'framework_match': np.mean([p['framework_match'] for p in pattern_scores]),
                'depth_potential': np.mean([p['depth_potential'] for p in pattern_scores]),
                'engagement_potential': np.mean([p['engagement_potential'] for p in pattern_scores]),
                'personal_connection': np.mean([p['personal_connection'] for p in pattern_scores])
            }
        }
        print(f"✅ Baselines established from your core questions")
        return baselines
    
    def _calculate_pattern_scores(self, question: str) -> Dict[str, float]:
        scores = {'framework_match': 0.0, 'depth_potential': 0.0, 'engagement_potential': 0.0, 'personal_connection': 0.0}
        q_lower = question.lower()
        framework_patterns = {
            'what': r'^what\\s+(is|was|are|were|would|do|did)', 'how': r'^how\\s+(do|did|would|can|could)',
            'if': r'(if you|imagine|suppose)', 'history': r'(have you ever|when was|tell me about)',
            'prefs': r'(favorite|prefer|like most|enjoy most)',
        }
        scores['framework_match'] = sum(1 for p in framework_patterns.values() if re.search(p, q_lower)) * 20
        scores['depth_potential'] = sum(1 for i in ['why','how','meaning','purpose','believe','think','feel'] if i in q_lower) * 15
        if re.search(r'(if you|imagine|suppose|what if)', q_lower): scores['depth_potential'] += 25
        scores['engagement_potential'] = sum(1 for w in ['favorite','best','most','ever','always','never','love'] if w in q_lower) * 20
        scores['personal_connection'] = min(60, sum(q_lower.count(w) for w in ['you','your','yourself','personal','own','life']) * 10)
        return scores
    
    def calculate_similarity_score(self, question: str) -> float:
        try:
            q_vec = self.vectorizer.transform([question])
            sims = cosine_similarity(q_vec, self.core_vectors)[0]
            return float(0.7 * np.max(sims) + 0.3 * np.mean(np.sort(sims)[-5:]))
        except: return 0.0
    
    def calculate_hybrid_score(self, question: str, weights: Dict[str, float] = None) -> Dict[str, float]:
        """Calculate comprehensive hybrid score (Simplified version)."""
        if weights is None: weights = HYBRID_CONFIG # Use the global config
        
        sim_raw = self.calculate_similarity_score(question)
        p_scores = self._calculate_pattern_scores(question)
        
        b_sim = self.baselines['similarity_baseline']
        b_patt = self.baselines['pattern_baselines']
        
        sim_cal = 60 + (sim_raw - b_sim) / (1.0 - b_sim) * 40 if sim_raw >= b_sim else (sim_raw / b_sim) * 60 if b_sim > 0 else 0
        
        cal_patt = {}
        for key, raw_score in p_scores.items():
            baseline = b_patt.get(key, 20)
            cal_patt[key] = 60 + min(40, (raw_score - baseline) / baseline * 40) if raw_score >= baseline else (raw_score / baseline) * 60 if baseline > 0 else 0

        final_score = (
            sim_cal * weights['similarity_weight'] + cal_patt['framework_match'] * weights['framework_weight'] +
            cal_patt['depth_potential'] * weights['depth_weight'] + cal_patt['engagement_potential'] * weights['engagement_weight'] +
            cal_patt['personal_connection'] * weights['personal_weight']
        )
        final_score = min(100, max(0, final_score))

        return {
            'overall_score': round(final_score, 1), 'similarity_to_core': round(sim_cal, 1),
            'similarity_raw': round(sim_raw, 3), 'framework_match': round(cal_patt['framework_match'], 1),
            'depth_potential': round(cal_patt['depth_potential'], 1), 'engagement_potential': round(cal_patt['engagement_potential'], 1),
            'personal_connection': round(cal_patt['personal_connection'], 1)
        }

    def get_quality_assessment(self, score: float) -> str:
        if score >= 85: return "Excellent - Matches core app question quality"
        elif score >= 75: return "Very Good - Strong conversation potential"
        elif score >= 65: return "Good - Solid dating question"
        elif score >= 55: return "Fair - Decent with some potential"
        else: return "Poor - Not suitable for dating conversations"
        
    def find_similar_core_questions(self, question: str, top_n: int = 3) -> List[Tuple[str, float]]:
        try:
            q_vec = self.vectorizer.transform([question])
            sims = cosine_similarity(q_vec, self.core_vectors)[0]
            top_indices = np.argsort(sims)[-top_n:][::-1]
            return [(self.core_questions[i], round(sims[i], 3)) for i in top_indices]
        except: return []

print("🧠 Hybrid scoring model implementation loaded (with simplified scoring)!")

🧠 Hybrid scoring model implementation loaded!


## 🔍 Load and Initialize Your Core Questions

This loads your core questions and initializes the hybrid model.

In [26]:
def load_core_questions_from_file(filepath: str) -> List[str]:
    """Load and clean core questions from file"""
    questions = []
    
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            content = f.read()
        
        lines = content.split('\n')
        for line in lines:
            line = line.strip()
            if re.match(r'^\d+\.', line):
                question = re.sub(r'^\d+\.\s*', '', line).strip()
                if question and len(question) > 10:
                    questions.append(question)
        
        return questions
    
    except FileNotFoundError:
        print(f"❌ Core questions file not found: {filepath}")
        print("Please upload your core questions file and update the CORE_QUESTIONS_FILE path above.")
        return []
    except Exception as e:
        print(f"❌ Error loading core questions: {e}")
        return []

# Load your core questions
print("📚 Loading your core questions...")
core_questions = load_core_questions_from_file(CORE_QUESTIONS_FILE)

if core_questions:
    print(f"✅ Loaded {len(core_questions)} core questions")
    
    # Initialize hybrid model
    hybrid_model = HybridScoringModel(core_questions)
    
    # Show sample core questions
    print(f"\n📝 Sample core questions:")
    for i, q in enumerate(core_questions[:5], 1):
        print(f"   {i}. {q}")
    
    print(f"\n🎯 Hybrid model ready for question scoring!")
else:
    print("❌ No core questions loaded - please check the file path")
    hybrid_model = None

📚 Loading your core questions...
✅ Loaded 210 core questions
🔧 Analyzing your core questions to establish quality baselines...
✅ Baselines established from your core questions
🤖 Hybrid model initialized with 210 core questions
📊 Quality baselines established from your data

📝 Sample core questions:
   1. What is the smallest thing for which you are grateful?
   2. Who has had the most positive impact on your life?
   3. If you could use a time machine, would you rather have one that only goes back in time or only goes forward?
   4. If you got a promotion, a job, a college acceptance, an accolade/award, or just generally accomplished something major, who is the first person you'd tell and how do you think they'd react?
   5. If you were an inanimate object, what would you be and why?

🎯 Hybrid model ready for question scoring!


## 🧪 Test the Hybrid Model

Test how the hybrid model scores different types of questions.

In [27]:
def analyze_single_question(question: str):
    """Analyze a single question in detail"""
    if not hybrid_model:
        print("❌ Hybrid model not initialized")
        return
    
    print(f"🔍 Analyzing: '{question}'")
    print("=" * 60)
    
    # Calculate hybrid score
    scores = hybrid_model.calculate_hybrid_score(question)
    quality = hybrid_model.get_quality_assessment(scores['overall_score'])
    
    print(f"🎯 Overall Score: {scores['overall_score']:.1f}/100")
    print(f"📊 Quality Assessment: {quality}")
    print(f"\n📈 Score Breakdown:")
    print(f"   Similarity to core: {scores['similarity_to_core']:.1f}/100 (raw: {scores['similarity_raw']:.3f})")
    print(f"   Framework match: {scores['framework_match']:.1f}/100")
    print(f"   Depth potential: {scores['depth_potential']:.1f}/100")
    print(f"   Engagement potential: {scores['engagement_potential']:.1f}/100")
    print(f"   Personal connection: {scores['personal_connection']:.1f}/100")
    
    # Show most similar core questions
    similar = hybrid_model.find_similar_core_questions(question, 3)
    if similar:
        print(f"\n🔗 Most similar core questions:")
        for i, (core_q, sim_score) in enumerate(similar, 1):
            print(f"   {i}. [{sim_score:.3f}] {core_q}")
    
    return scores

# Test with sample questions
if hybrid_model:
    print("🧪 Testing Hybrid Model with Sample Questions")
    print("=" * 60)
    
    test_questions = [
        "What's your favorite way to spend a weekend?",
        "What's the most meaningful gift you've ever received?",
        "If you could change one thing about your past, what would it be?",
        "Do you like pizza?",
        "What's your job?"
    ]
    
    for question in test_questions:
        scores = analyze_single_question(question)
        print("\n" + "-" * 60 + "\n")
else:
    print("⚠️ Hybrid model not available - please load core questions first")

🧪 Testing Hybrid Model with Sample Questions
🔍 Analyzing: 'What's your favorite way to spend a weekend?'
🎯 Overall Score: 66.2/100
📊 Quality Assessment: Good - Solid dating question

📈 Score Breakdown:
   Similarity to core: 63.0/100 (raw: 0.302)
   Framework match: 100.0/100
   Depth potential: 0.0/100
   Engagement potential: 100.0/100
   Personal connection: 56.6/100

🔗 Most similar core questions:
   1. [0.322] What is your favorite word and why?
   2. [0.313] What is your ideal birthday? Not the date, but rather your ideal way to spend the day.
   3. [0.229] What is something you had to learn the hard way?

------------------------------------------------------------

🔍 Analyzing: 'What's the most meaningful gift you've ever received?'
🎯 Overall Score: 63.5/100
📊 Quality Assessment: Fair - Decent with some potential

📈 Score Breakdown:
   Similarity to core: 72.5/100 (raw: 0.482)
   Framework match: 0.0/100
   Depth potential: 100.0/100
   Engagement potential: 100.0/100
   Person

## 🎛️ Interactive Question Analysis

Analyze any question in real-time to see how the hybrid model scores it.

In [28]:
# Interactive question analysis
def interactive_analysis():
    """Interactive question analysis tool"""
    if not hybrid_model:
        print("❌ Hybrid model not initialized")
        return
    
    print("🎛️ Interactive Question Analysis")
    print("Enter questions to analyze (type 'quit' to exit)")
    print("=" * 50)
    
    while True:
        question = input("\n📝 Enter question: ").strip()
        
        if question.lower() in ['quit', 'exit', 'q']:
            print("👋 Goodbye!")
            break
        
        if not question:
            continue
        
        analyze_single_question(question)
        print("\n" + "-" * 50)

# Uncomment the line below to run interactive analysis
# interactive_analysis()

print("💡 Tip: Uncomment the line above to run interactive question analysis")

💡 Tip: Uncomment the line above to run interactive question analysis


## 🔧 Parameter Tuning and Optimization

Adjust the hybrid model parameters to optimize for your specific needs.

In [29]:
def test_different_weights(question: str, weight_configs: List[Dict[str, float]]):
    """Test how different weight configurations affect scoring"""
    if not hybrid_model:
        print("❌ Hybrid model not initialized")
        return
    
    print(f"🔧 Testing weight configurations for: '{question}'")
    print("=" * 60)
    
    for i, weights in enumerate(weight_configs, 1):
        scores = hybrid_model.calculate_hybrid_score(question, weights)
        quality = hybrid_model.get_quality_assessment(scores['overall_score'])
        
        print(f"\n🎯 Configuration {i}: Score {scores['overall_score']:.1f} - {quality}")
        print(f"   Weights: {weights}")
        print(f"   Breakdown: Sim={scores['similarity_to_core']:.1f}, Frame={scores['framework_match']:.1f}, Depth={scores['depth_potential']:.1f}")

# Test different weight configurations
if hybrid_model:
    test_question = "What's something you believed as a child that you later realized wasn't true?"
    
    weight_configs = [
        # Current configuration
        {
            'similarity_to_core': 0.40,
            'framework_match': 0.25,
            'depth_potential': 0.20,
            'engagement_potential': 0.10,
            'personal_connection': 0.05
        },
        # Similarity-focused
        {
            'similarity_to_core': 0.60,
            'framework_match': 0.15,
            'depth_potential': 0.15,
            'engagement_potential': 0.05,
            'personal_connection': 0.05
        },
        # Depth-focused
        {
            'similarity_to_core': 0.25,
            'framework_match': 0.20,
            'depth_potential': 0.35,
            'engagement_potential': 0.15,
            'personal_connection': 0.05
        },
        # Balanced
        {
            'similarity_to_core': 0.30,
            'framework_match': 0.25,
            'depth_potential': 0.25,
            'engagement_potential': 0.15,
            'personal_connection': 0.05
        }
    ]
    
    test_different_weights(test_question, weight_configs)
else:
    print("⚠️ Hybrid model not available")

🔧 Testing weight configurations for: 'What's something you believed as a child that you later realized wasn't true?'

🎯 Configuration 1: Score 61.8 - Fair - Decent with some potential
   Weights: {'similarity_to_core': 0.4, 'framework_match': 0.25, 'depth_potential': 0.2, 'engagement_potential': 0.1, 'personal_connection': 0.05}
   Breakdown: Sim=77.3, Frame=0.0, Depth=100.0

🎯 Configuration 2: Score 73.9 - Good - Solid dating question
   Weights: {'similarity_to_core': 0.6, 'framework_match': 0.15, 'depth_potential': 0.15, 'engagement_potential': 0.05, 'personal_connection': 0.05}
   Breakdown: Sim=77.3, Frame=0.0, Depth=100.0

🎯 Configuration 3: Score 65.7 - Good - Solid dating question
   Weights: {'similarity_to_core': 0.25, 'framework_match': 0.2, 'depth_potential': 0.35, 'engagement_potential': 0.15, 'personal_connection': 0.05}
   Breakdown: Sim=77.3, Frame=0.0, Depth=100.0

🎯 Configuration 4: Score 58.7 - Fair - Decent with some potential
   Weights: {'similarity_to_core': 0.3,

## 🎭 Demo Mode - Test Without Reddit API

Run a complete demo extraction using sample questions to test the hybrid scoring.

In [30]:
def run_hybrid_demo():
    """Run hybrid demo with sample questions"""
    if not hybrid_model:
        print("❌ Hybrid model not initialized")
        return None
    
    print("🎭 Running Hybrid Demo - Testing Sample Questions")
    print("=" * 60)
    
    # Sample questions of varying quality
    demo_questions = [
        "What's your favorite way to spend a weekend?",
        "What's the most meaningful gift you've ever received?",
        "If you could change one thing about your past, what would it be?",
        "What's something you believed as a child that you later realized wasn't true?",
        "What's a memory that always makes you smile?",
        "What's the biggest risk you've ever taken?",
        "What's something you're passionate about that others might find boring?",
        "What's the best advice you've ever received?",
        "If you could master any skill instantly, what would it be?",
        "What's something that always puts you in a good mood?",
        "What's your biggest fear and why?",
        "What's the most spontaneous thing you've ever done?",
        "What's something you wish you could tell your younger self?",
        "What's your idea of a perfect day?",
        "What's something you've learned about yourself recently?",
        "Do you like pizza?",  # Low quality
        "What's your job?",    # Low quality
        "How old are you?",   # Low quality
        "What's your name?",  # Low quality
        "Where are you from?" # Low quality
    ]
    
    # Score all questions
    results = []
    print(f"🎯 Scoring {len(demo_questions)} demo questions...")
    
    for i, question in enumerate(demo_questions):
        scores = hybrid_model.calculate_hybrid_score(question)
        quality = hybrid_model.get_quality_assessment(scores['overall_score'])
        
        # Find most similar core question
        similar = hybrid_model.find_similar_core_questions(question, 1)
        most_similar = similar[0][0] if similar else "None found"
        similarity_score = similar[0][1] if similar else 0.0
        
        results.append({
            'question_id': f'DEMO{i+1:03d}',
            'question': question,
            'theme': 'demo',
            'reddit_topic': 'Demo',
            'score': scores['overall_score'],
            'timestamp': datetime.now().isoformat(),
            'similarity_to_core': scores['similarity_to_core'],
            'framework_match': scores['framework_match'],
            'depth_potential': scores['depth_potential'],
            'engagement_potential': scores['engagement_potential'],
            'personal_connection': scores['personal_connection'],
            'quality_assessment': quality,
            'most_similar_core': most_similar,
            'similarity_raw': similarity_score,
            'reddit_score': 100,
            'reddit_comments': 25,
            'source': 'demo'
        })
    
    # Filter by quality threshold
    high_quality = [r for r in results if r['score'] >= HYBRID_CONFIG['min_score_threshold']]
    
    # Sort by score
    high_quality.sort(key=lambda x: x['score'], reverse=True)
    
    # Create DataFrame
    df = pd.DataFrame(high_quality)
    
    # Export to Excel
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"hybrid_demo_questions_{timestamp}.xlsx"
    filepath = csv_dir / filename
    
    df.to_excel(filepath, index=False)
    
    # Print results
    print(f"\n✅ Demo completed successfully!")
    print(f"📊 Results:")
    print(f"   Total questions tested: {len(demo_questions)}")
    print(f"   Questions above threshold ({HYBRID_CONFIG['min_score_threshold']}): {len(high_quality)}")
    print(f"   Average score (high quality): {np.mean([r['score'] for r in high_quality]):.1f}")
    print(f"   Excel file: {filepath}")
    
    # Show top questions
    print(f"\n🌟 Top 5 Questions by Hybrid Score:")
    for i, result in enumerate(high_quality[:5], 1):
        print(f"   {i}. [{result['score']:5.1f}] {result['question']}")
        print(f"      Most similar core: {result['most_similar_core'][:60]}...")
    
    # Show quality distribution
    excellent = len([r for r in high_quality if r['score'] >= 85])
    very_good = len([r for r in high_quality if 75 <= r['score'] < 85])
    good = len([r for r in high_quality if 65 <= r['score'] < 75])
    fair = len([r for r in high_quality if 55 <= r['score'] < 65])
    
    print(f"\n📈 Quality Distribution:")
    print(f"   Excellent (≥85): {excellent}")
    print(f"   Very Good (75-84): {very_good}")
    print(f"   Good (65-74): {good}")
    print(f"   Fair (55-64): {fair}")
    
    return df, filepath

# Run the demo
if hybrid_model:
    demo_results = run_hybrid_demo()
else:
    print("⚠️ Please load core questions first to run the demo")

🎭 Running Hybrid Demo - Testing Sample Questions
🎯 Scoring 20 demo questions...



✅ Demo completed successfully!
📊 Results:
   Total questions tested: 20
   Questions above threshold (60): 5
   Average score (high quality): 68.6
   Excel file: /Users/justsuyash/Documents/GitHub/AlignedV1.001/notebooks/outputs/csv/hybrid_demo_questions_20250617_191702.xlsx

🌟 Top 5 Questions by Hybrid Score:
   1. [ 80.7] If you could change one thing about your past, what would it be?
      Most similar core: If you could change one thing about yourself physically, wha...
   2. [ 71.0] If you could master any skill instantly, what would it be?
      Most similar core: What instantly makes you feel like a child again?...
   3. [ 66.2] What's your favorite way to spend a weekend?
      Most similar core: What is your favorite word and why?...
   4. [ 63.5] What's the most meaningful gift you've ever received?
      Most similar core: What's the best gift you've ever received? Who gave it to yo...
   5. [ 61.8] What's something you believed as a child that you later realized wasn't tr

## 🚀 Live Reddit Extraction and Hybrid Scoring

This section combines the Reddit extractor with the hybrid scoring model to create a complete, live pipeline. It replaces the previous demo mode.

1.  **Extracts** questions from the subreddits defined in `SUBREDDITS`.
2.  **Scores** each extracted question using the `HybridScoringModel`.
3.  **Filters** out questions that don't meet the `min_score_threshold`.
4.  **Exports** the high-quality results to a timestamped Excel file.

In [ ]:
import time
import pandas as pd

def collect_raw_questions(reddit_instance, subreddits_list, config):
    """
    Connects to Reddit and extracts a raw list of potential questions.
    """
    if not reddit_instance:
        print("❌ Reddit connection not available. Cannot proceed.")
        return []

    print("🚀 STEP 1: Starting Data Collection from Reddit")
    print("=" * 60)

    extractor = RedditQuestionExtractor(reddit_instance)
    all_raw_questions = []

    for subreddit in subreddits_list:
        raw_questions = extractor.extract_from_subreddit(
            subreddit_name=subreddit,
            posts_limit=config['posts_per_subreddit']
        )
        all_raw_questions.extend(raw_questions)
        time.sleep(1) # Be respectful to the API

    print(f"\n✅ Collection complete. Found {len(all_raw_questions)} total potential questions.")
    print("   The collected data is now stored in the 'raw_questions_data' variable.")
    return all_raw_questions

# --- Execute Collection Step ---
# This variable will hold the output of this step
raw_questions_data = collect_raw_questions(reddit, SUBREDDITS, HYBRID_CONFIG)

🚀 STEP 1: Starting Data Collection from Reddit
🔭 Searching for questions in r/dating_advice...


Processing r/dating_advice:   0%|          | 0/30 [00:00<?, ?it/s]

   - Found 29 new potential questions.
🔭 Searching for questions in r/dating...


Processing r/dating:   0%|          | 0/30 [00:00<?, ?it/s]

   - Found 22 new potential questions.
🔭 Searching for questions in r/relationships...


...

   - Found 31 new potential questions.
🔭 Searching for questions in r/socialskills...


Processing r/socialskills:   0%|          | 0/30 [00:00<?, ?it/s]

   - Found 43 new potential questions.
🔭 Searching for questions in r/DeepThoughts...


Processing r/DeepThoughts:   0%|          | 0/30 [00:00<?, ?it/s]

   - Found 47 new potential questions.

✅ Collection complete. Found 388 total potential questions.
   The collected data is now stored in the 'raw_questions_data' variable.


In [ ]:
# ==============================================================================
# 🧠 STEP 3: SCORE COLLECTED QUESTIONS
# ==============================================================================

def score_collected_questions(raw_data, model, config):
    """
    Scores a list of raw questions using the hybrid model and filters by score.
    """
    if not raw_data:
        print("❌ No raw questions to score. Please run the collection step first.")
        return []

    print("\n🧠 Scoring Collected Questions...")
    print(f"   - Minimum score threshold: {config['min_score_threshold']}")
    print("=" * 60)

    scored_results = []
    
    for item in tqdm(raw_data, desc="Scoring questions"):
        # The 'item' dictionary already contains the question and its metadata
        scores = model.calculate_hybrid_score(item['question'])

        # Keep the question if it meets the minimum score threshold
        if scores['overall_score'] >= config['min_score_threshold']:
            # Merge the original item data (question, subreddit, etc.) with the new scores
            item.update(scores)
            scored_results.append(item)
            
    print(f"\n✅ Scoring complete. {len(scored_results)} questions met the quality threshold.")
    print("   The scored data is now stored in the 'scored_questions_data' variable.")
    return scored_results

# --- Execute Scoring Step ---
# This uses the output from the previous step ('raw_questions_data') as its input
# and the HYBRID_CONFIG dictionary for the score threshold.
scored_questions_data = score_collected_questions(raw_questions_data, hybrid_model, HYBRID_CONFIG)


🧠 STEP 2: Scoring Collected Questions


Scoring questions:   0%|          | 0/388 [00:00<?, ?it/s]


✅ Scoring complete. 36 questions met the quality threshold of 60.
   The scored data is now stored in the 'scored_questions_data' variable.


In [33]:
import hashlib
from datetime import datetime
from IPython.display import display

def assign_topic_label(question_text: str) -> str:
    """Assigns a topic/label to a question based on keywords."""
    question_lower = question_text.lower()
    if any(k in question_lower for k in ['if you', 'what if', 'imagine', 'would you']): return 'Hypotheticals'
    if any(k in question_lower for k in ['you', 'your', 'yourself']): return 'About You'
    if any(k in question_lower for k in ['date', 'perfect day', 'romantic', 'together']): return 'Date Vibe'
    if any(k in question_lower for k in ['hobby', 'passion', 'spend your time', 'weekend', 'job', 'work']): return 'Lifestyle'
    if any(k in question_lower for k in ['memory', 'childhood', 'dream', 'goal', 'in the future']): return 'Past & Future'
    if any(k in question_lower for k in ['i', 'me', 'my']): return 'About Me'
    return 'General'

def create_and_save_dataframe(scored_data):
    """
    Creates the final structured DataFrame and saves it to an Excel file.
    """
    if not scored_data:
        print("❌ No scored data to process. Please run the scoring step first.")
        return None

    print("\n📋 STEP 3: Creating and Saving Final DataFrame")
    print("=" * 60)
    
    final_data_list = []
    for item in scored_data:
        final_data_list.append({
            'quesiton_id': hashlib.md5(item['question'].encode()).hexdigest(),
            'question': item['question'],
            'topic/label': assign_topic_label(item['question']),
            'score': item.get('overall_score'),
            'similarity_to_core': item.get('similarity_to_core'),
            'framework_match': item.get('framework_match'),
            'depth_potential': item.get('depth_potential'),
            'engagement_potential': item.get('engagement_potential'),
            'personal_connection': item.get('personal_connection'),
            'reddit_topic': item['reddit_topic'],
            'timestamp': datetime.now().isoformat()
        })
        
    # Create the DataFrame
    results_df = pd.DataFrame(final_data_list)
    results_df.sort_values(by="score", ascending=False, inplace=True)
    results_df.drop_duplicates(subset=['question'], keep='first', inplace=True)

    # Save the file
    outputs_dir.mkdir(parents=True, exist_ok=True)
    filepath = outputs_dir / "Questions.xlsx"
    results_df.to_excel(filepath, index=False)
    
    print("\n🎉 Pipeline Finished Successfully!")
    print(f"   - Final DataFrame created with {len(results_df)} questions.")
    print(f"   - Saved to file: {filepath}")
    
    print("\n🔍 Final DataFrame Preview:")
    display(results_df.head())
    return results_df

# --- Execute DataFrame Creation Step ---
# This uses the output from Step 2 ('scored_questions_data') as its input
final_df = create_and_save_dataframe(scored_questions_data)


📋 STEP 3: Creating and Saving Final DataFrame

🎉 Pipeline Finished Successfully!
   - Final DataFrame created with 36 questions.
   - Saved to file: /Users/justsuyash/Documents/GitHub/AlignedV1.001/notebooks/outputs/Questions.xlsx

🔍 Final DataFrame Preview:


,quesiton_id,question,topic/label,score,similarity_to_core,framework_match,depth_potential,engagement_potential,personal_connection,reddit_topic,timestamp
2,dbb68fbaf1fbc5b4323c2289843a2e78,If you’re always the one initiating the text c...,Hypotheticals,100.0,74.8,100.0,100.0,100.0,56.6,Giving Advice 💌,2025-06-17T19:17:21.030287
3,b7f300bafc73aa7774ea26c44f06592b,If you’re always initiating the hangouts and ...,Hypotheticals,91.2,72.6,100.0,100.0,100.0,56.6,Giving Advice 💌,2025-06-17T19:17:21.030291
4,33a589ed0b4306bc80bee85e29a318a0,"If you are ever taken advantage of, I will kn...",Hypotheticals,89.5,68.6,100.0,100.0,100.0,56.6,N/A,2025-06-17T19:17:21.030294
27,b0206222dc631a6e163314d4d33d1f20,How do I tactfully tell a girl I don't want to...,About Me,85.9,74.2,100.0,100.0,0.0,0.0,N/A,2025-06-17T19:17:21.030382
30,08ce82b0e2a490218121bdc632ddddd9,help!! accidentally invited 7 people to a hang...,About Me,84.9,64.7,100.0,100.0,100.0,0.0,N/A,2025-06-17T19:17:21.030396


## 📊 Results Analysis and Insights

Analyze the results from the hybrid scoring to understand patterns and quality.

In [34]:
def analyze_hybrid_results(df: pd.DataFrame, source_name: str):
    """Analyze results from hybrid scoring"""
    if df is None or len(df) == 0:
        print(f"❌ No {source_name} results to analyze.")
        return
    
    print(f"📊 Hybrid Results Analysis for: {source_name.upper()} DATA")
    print("=" * 60)
    
    # Basic statistics
    print(f"📈 Score Statistics:")
    print(f"   Mean: {df['score'].mean():.1f}")
    print(f"   Median: {df['score'].median():.1f}")
    print(f"   Std Dev: {df['score'].std():.1f}")
    print(f"   Range: {df['score'].min():.1f} - {df['score'].max():.1f}")
    
    # Component analysis
    print(f"\n🔍 Component Analysis:")
    # Use the columns that are actually in the final DataFrame
    components = ['similarity_to_core', 'framework_match', 'depth_potential', 
                 'engagement_potential', 'personal_connection']
    
    for component in components:
        if component in df.columns:
            mean_score = df[component].mean()
            print(f"   {component.replace('_', ' ').title()}: {mean_score:.1f}")
    
    # Quality distribution
    print(f"\n🏆 Quality Distribution:")
    excellent = len(df[df['score'] >= 85])
    very_good = len(df[(df['score'] >= 75) & (df['score'] < 85)])
    good = len(df[(df['score'] >= 65) & (df['score'] < 75)])
    
    total = len(df)
    if total > 0:
        print(f"   Excellent (≥85): {excellent} ({excellent/total*100:.1f}%)")
        print(f"   Very Good (75-84): {very_good} ({very_good/total*100:.1f}%)")
        print(f"   Good (65-74): {good} ({good/total*100:.1f}%)")

    # Top performers
    print(f"\n🌟 Top 3 Questions:")
    top_3 = df.nlargest(3, 'score')
    for i, (_, row) in enumerate(top_3.iterrows(), 1):
        question_text = row['question']
        print(f"   {i}. [{row['score']:5.1f}] {question_text}")

# ===================================================================
# STEP 4: PERFORM DETAILED RESULTS ANALYSIS (OPTIONAL)
# ===================================================================
print("\n🔎 STEP 4: Performing Detailed Results Analysis")
print("=" * 60)

# This now uses the 'final_df' variable created at the end of Step 3
if 'final_df' in locals() and isinstance(final_df, pd.DataFrame):
    analyze_hybrid_results(final_df, source_name="Final")
else:
    print("❌ No final DataFrame found to analyze. Please run Step 3 first.")


🔎 STEP 4: Performing Detailed Results Analysis
📊 Hybrid Results Analysis for: FINAL DATA
📈 Score Statistics:
   Mean: 73.1
   Median: 71.0
   Std Dev: 9.8
   Range: 61.1 - 100.0

🔍 Component Analysis:
   Similarity To Core: 64.7
   Framework Match: 75.0
   Depth Potential: 94.4
   Engagement Potential: 50.0
   Personal Connection: 37.0

🏆 Quality Distribution:
   Excellent (≥85): 4 (11.1%)
   Very Good (75-84): 11 (30.6%)
   Good (65-74): 12 (33.3%)

🌟 Top 3 Questions:
   1. [100.0] If you’re always the one initiating the text convos she doesn’t want you
   2. [ 91.2]  If you’re always initiating the hangouts and it always seems like she’s stalling out or coming up with an excuse she doesn’t want you
   3. [ 89.5]  If you are ever taken advantage of, I will know on some level you consented


## 🚀 Complete Documentation and Usage Guide

### 🎯 Hybrid Model Overview

This hybrid scoring model represents a significant advancement over traditional keyword-based approaches. Instead of relying on generic patterns, it learns directly from your proven high-quality questions to identify similar content on Reddit.

#### 🧠 How It Works

1. **Core Question Analysis**: The model analyzes your 300+ core questions to understand:
   - Question frameworks and structures
   - Emotional depth indicators
   - Engagement patterns
   - Personal connection elements
   - Conversation flow potential

2. **Baseline Establishment**: It establishes quality baselines by analyzing how your core questions score against each other, creating a calibrated understanding of what "high quality" means for your specific use case.

3. **Similarity Matching**: Uses TF-IDF vectorization and cosine similarity to find Reddit questions that are semantically similar to your core questions.

4. **Pattern Recognition**: Identifies specific patterns that make questions effective:
   - Question frameworks ("What if...", "How do you...", etc.)
   - Depth indicators ("why", "meaning", "purpose")
   - Engagement triggers ("favorite", "best", "most")
   - Personal connection words ("you", "your", "personal")

5. **Calibrated Scoring**: Scores questions on a 0-100 scale that's calibrated to your quality standards, not generic benchmarks.

#### 📊 Scoring Components

- **Similarity to Core (40%)**: How similar the question is to your proven questions
- **Framework Match (25%)**: How well it matches proven question frameworks
- **Depth Potential (20%)**: Potential for deep, meaningful conversation
- **Engagement Potential (10%)**: Likelihood to engage and interest users
- **Personal Connection (5%)**: Potential for personal sharing and connection

#### 🎯 Quality Thresholds

- **Excellent (85-100)**: Matches core app question quality
- **Very Good (75-84)**: Strong conversation potential
- **Good (65-74)**: Solid dating question
- **Fair (55-64)**: Decent with some potential
- **Below Average (<55)**: Limited value

#### 🔧 Customization Options

- **Weight Adjustment**: Modify the importance of different scoring components
- **Threshold Tuning**: Adjust quality thresholds based on your needs
- **Subreddit Selection**: Choose which subreddits to extract from
- **Dating Relevance**: Toggle dating-specific filtering
- **Similarity Sensitivity**: Adjust how strict similarity matching should be

#### 💡 Best Practices

1. **Start with Higher Thresholds**: Begin with a threshold of 70+ for better quality
2. **Monitor Component Scores**: Look at the breakdown to understand why questions score well
3. **Use Similar Core Questions**: Check which core questions are most similar to understand the match
4. **Adjust Weights Based on Goals**: Increase similarity weight for closer matches to your style
5. **Regular Retraining**: As you add more core questions, retrain the model for better accuracy

#### 🚀 Advanced Usage

- **A/B Testing**: Test different weight configurations to optimize for your specific goals
- **Quality Analysis**: Use the detailed breakdowns to understand what makes questions effective
- **Continuous Improvement**: Analyze results to refine your approach over time
- **Custom Filtering**: Add additional filters based on your specific requirements

This hybrid approach ensures that the questions you extract from Reddit are not just generically "good" but specifically aligned with the proven quality and style of your core app questions.